# Physical projection: 2601_XPCS
Run 00_prepare first. Shared cell structure for both paper cases.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import h5py
import matplotlib.pyplot as plt
from copy import deepcopy
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/source_manifest.json").is_file())
sys.path.insert(0, str(ROOT))
from library import phase_retrieval_universal as universal
from library import phase_retrieval_geometry as geometry
CACHE = ROOT / "data/2601_XPCS/prepared.h5"
OUTPUT = ROOT / "results/hysteresis.h5"
SATURATED_POINTS = {}  # e.g. {(263, 10): +1., (263, 34): -1.}; verify independently.
POINT_INDICES = [10,12,14,15,17,20,23,27,28,29,30,31,32,34]
with h5py.File(CACHE) as handle:
    config=json.loads(handle.attrs['config_json'])
    labels=handle['state_labels'].asstr()[:].tolist()
    keys=list(zip(handle['scan'][()].tolist(),handle['point'][()].tolist()))
    indices=list(POINT_INDICES) if POINT_INDICES is not None else list(range(len(labels)))
    for key in SATURATED_POINTS:
        if key not in keys: raise ValueError(f"Missing anchor {key}")
        if keys.index(key) not in indices: indices.append(keys.index(key))
    if len(set(indices)) != len(indices) or any(i<0 or i>=len(labels) for i in indices):
        raise ValueError("Invalid acquisition indices")
    indices=sorted(indices)  # Preserve acquisition order, including repeated currents.
    holograms=np.stack([handle['holograms'][i] for i in indices])
    mask_pixel=np.stack([handle['mask_pixel'][i] for i in indices])
    support=handle['supportmask'][()]; material=handle['material_mask'][()].astype(float)
    current=handle['current_A'][()][indices]
    energies=handle['energy_ev'][()][indices]
    state_labels=[labels[i] for i in indices]
if np.ptp(energies)>.25: raise ValueError("Expected one photon energy")
energy_labels=[float(np.median(energies))]*len(indices)
polarizations=[1.]*len(indices)
saturated={labels[keys.index(key)]:value for key,value in SATURATED_POINTS.items()}
crop, binning=410,1  # Applied once by the reconstruction geometry.


## 1. Validate inputs and metadata

In [ ]:
assert holograms.ndim==3 and mask_pixel.shape==holograms.shape
assert support.shape==holograms.shape[-2:] and material.shape==support.shape
assert len(state_labels)==len(energy_labels)==len(polarizations)==len(holograms)
assert np.all(np.isfinite(holograms)) and np.any(support)
assert set(saturated.values()) <= {-1.,1.}
illumination_labels=["beam0"]*len(holograms)
metadata=dict(state_labels=state_labels, energy_labels=energy_labels,
              polarization_coefficients=polarizations, illumination_labels=illumination_labels)
print("Observations",len(holograms),"energies",sorted(set(energy_labels)),"anchors",saturated)
fig,ax=plt.subplots(1,2)
ax[0].imshow(np.log1p(np.maximum(holograms[0],0))); ax[0].set_title('First detector image')
ax[1].imshow(support); ax[1].set_title('Source support')


## 2. Set the physical model and iteration schedule

In [ ]:
# Shared controls. Change these explicitly for a new dataset.
MODES=[1,2]
OUTER_ITERATIONS=20
INNER_ITERATIONS=50
PHYSICAL_ITERATIONS=50
recipe=universal.default_universal_phase_retrieval_recipe()
recipe.update(projection_model="physical_factorized", modes=MODES,
    crop=crop, binning=binning, roi=None,
    recenter_modal_supports=True, mode_support_center="image",
    material_thickness=material, fit_material_thickness=False,
    physical_projection_object_roi=True, physical_phase_reference=True,
    saturated_states=saturated or None,
    mode_initialization="support_fft", mode_initialization_seed=0,
    partial_coherence=False,
    inner_mode="ER", inner_Nit=INNER_ITERATIONS,
    outer_iterations=OUTER_ITERATIONS, physical_iterations=PHYSICAL_ITERATIONS,
    projection_every=len(holograms), projection_relaxation=1.,
    final_projection_relaxation=1., final_fourier_constraint=False,
    constrain_nonphysical_modes_common=True,
    projection_constraints_inside_support_only=True, zero_magnetization_outside_support=True,
    clip_magnetization=False, magnetization_trend="none", bound_magnetization_median=False,
    physical_spatial_weighting="exit_wave_intensity", random_seed=0,
    observation_workers=1, fft_workers=1)
# Final model fields satisfy the physical projection; detector agreement is measured below.
recipe.update(energy_values=np.asarray(sorted(set(energy_labels))),
    align_global_phase=True, fit_illumination_intensity=True,
    fit_aperture_illumination=True, object_illumination_mode="linear_plane")


## 3. Independent phase retrieval warmup

In [ ]:
warmup_recipe=deepcopy(recipe)
warmup_recipe.update(projection_model="none", outer_iterations=0,
    final_projection_relaxation=0., final_fourier_constraint=False,
    constrain_nonphysical_modes_common=False, fit_illumination_intensity=False,
    fit_aperture_illumination=False, align_global_phase=False)
_, warmup, _, warm_masks, warm_errors=universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, support, **metadata, universal_recipe=warmup_recipe,
    phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel)
# Snapshot makes later accidental changes detectable before continuation.
import hashlib
warmup_signature=hashlib.sha256(holograms.tobytes()+mask_pixel.tobytes()+support.tobytes()+
    json.dumps(metadata,sort_keys=True).encode()+repr((crop,binning,MODES)).encode()).hexdigest()
warmup=warmup.copy(); warmup.setflags(write=False)


## 4. Alternate physical projection and phase retrieval

In [ ]:
signature=hashlib.sha256(holograms.tobytes()+mask_pixel.tobytes()+support.tobytes()+
    json.dumps(metadata,sort_keys=True).encode()+repr((crop,binning,MODES)).encode()).hexdigest()
if signature != warmup_signature: raise RuntimeError("Inputs changed; rerun warmup")
physical_recipe=deepcopy(recipe)
# The API accepts starts on the source detector grid. Embed the cached retrieval fields.
expanded=np.repeat(np.repeat(warmup,binning,axis=-2),binning,axis=-1)
origin=tuple(crop+(n-2*crop-u)//2 for n,u in zip(holograms.shape[-2:],expanded.shape[-2:]))
starts=np.zeros((*warmup.shape[:-2],*holograms.shape[-2:]),dtype=warmup.dtype)
starts[...,origin[0]:origin[0]+expanded.shape[-2],origin[1]:origin[1]+expanded.shape[-1]]=expanded
del expanded
measured, excluded, support_used, recovered, grid=geometry.prepare(
    holograms,mask_pixel,support,dict(physical_recipe,Nmodes=1),starts)
np.testing.assert_allclose(recovered,warmup,rtol=1e-12,atol=0)
del recovered
fields, received, components, bsmasks, errors=universal.universal_phase_retrieval_algorithm(
    holograms,mask_pixel,support,**metadata, universal_recipe=physical_recipe,start_fields=starts,
    phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel)
np.testing.assert_allclose(received,warmup,rtol=1e-12,atol=0)
del starts,received
print("Physical model residual",components.get('fit_residual_rms'))
print("Identifiability",components.get('identifiability', 'See model calibration documentation'))


## 5. Compare detector residuals and inspect maps

In [ ]:
# Compare every observation on valid detector pixels, using the sum over incoherent modes.
detector_errors={}
for label,stack in [('warmup',warmup),('physical',fields)]:
    prediction=np.sum(abs(stack)**2,axis=1) if stack.ndim==4 else abs(stack)**2
    values=[]
    for i in range(len(measured)):
        valid=~np.asarray(bsmasks[i],bool)
        denominator=np.linalg.norm(measured[i][valid])
        values.append(np.linalg.norm((prediction[i]-measured[i])[valid])/denominator if denominator else np.nan)
    detector_errors[label]=np.asarray(values)
fig,ax=plt.subplots()
for label,values in detector_errors.items(): ax.plot(values,'o-',label=label)
ax.set(xlabel='Observation in acquisition order',ylabel='Relative intensity residual'); ax.legend()
magnetization=np.asarray(components['magnetization'])
fig,ax=plt.subplots(); im=ax.imshow(magnetization[0],cmap='RdBu_r'); fig.colorbar(im,ax=ax)
ax.set_title('First fitted state: magnetization factor')


## 6. Extract the paper observables

In [ ]:
# Object maps are in the solver's centered physical-projection coordinates.
material_used=np.fft.fftshift(universal._recipe_material_thickness(physical_recipe,grid))>0
ROI=material_used  # Replace with a documented boolean ROI for local hysteresis loops.
if not np.any(ROI): raise ValueError('Empty analysis ROI')
state_names=list(components['state_names'])
state_currents=np.asarray([current[state_labels.index(name)] for name in state_names])
fit_curve=np.mean(magnetization[:,ROI],axis=1)
# Same endpoint calibration and ROI for warmup and final fields; no fitted monotonic trend.
objects=[]
for stack in [warmup,fields]:
    primary=stack[:,0] if stack.ndim==4 else stack
    objects.append(np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(primary,axes=(-2,-1)),axes=(-2,-1)),axes=(-2,-1)))
positive=next((i for i,s in enumerate(state_labels) if saturated.get(s)==1),None)
negative=next((i for i,s in enumerate(state_labels) if saturated.get(s)==-1),None)
paper_analysis={'current_A':state_currents,'model_curve':fit_curve,'roi_mask':ROI}
fig,ax=plt.subplots()
ax.plot(state_currents,fit_curve,'o-',label='Physical model')
if positive is not None and negative is not None:
    # Fixed complex contrast direction from independent-warmup endpoints.
    direction=np.mean((objects[0][positive]-objects[0][negative])[ROI])
    if not np.isfinite(direction) or abs(direction) <= 1e-12:
        print('Skipping endpoint normalization: endpoint contrast is degenerate for this calibration.')
    else:
        for label,obj in zip(['warmup','physical'],objects):
            signal=np.real(obj*np.conj(direction)/abs(direction))
            mean=np.mean(signal[:,ROI],axis=1)
            span=mean[positive]-mean[negative]
            if not np.isfinite(span) or abs(span) <= 1e-12:
                print(f'Skipping {label} endpoint normalization: span is degenerate.')
                continue
            curve=2*(mean-(mean[positive]+mean[negative])/2)/span
            paper_analysis[label+'_endpoint_curve']=curve
            ax.plot(state_currents,curve,'o--',label=label+' endpoint contrast')
else:
    print('No two-sign saturation calibration: model curve is a relative factor.')
ax.set(xlabel='Magnet current (A)',ylabel='Magnetization / relative contrast'); ax.legend()
# Current is not converted to field without an independently measured calibration.


## 7. Save data, settings, and the figure

In [ ]:
# Store arrays and nested numerical metadata directly; this small writer stays visible here.
def save_tree(group, values):
    for key,value in values.items():
        if isinstance(value,dict): save_tree(group.create_group(str(key)),value)
        elif value is None: group.attrs[str(key)]='None'
        else:
            array=np.asarray(value)
            if array.dtype.kind in 'biufc': group.create_dataset(str(key),data=array)
            else: group.attrs[str(key)]=json.dumps(value,default=lambda x: np.asarray(x).tolist())
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
if OUTPUT.exists(): raise FileExistsError(f'Choose a new OUTPUT path: {OUTPUT}')
with h5py.File(OUTPUT,'x') as handle:
    for name,value in dict(fields=fields,warmup_fields=warmup,holograms=measured,
                           mask_pixel=bsmasks,supportmask=support_used).items():
        handle.create_dataset(name,data=value,compression='gzip')
    save_tree(handle.create_group('metadata'),metadata)
    save_tree(handle.create_group('preprocessing'),config)
    save_tree(handle.create_group('recipe'),physical_recipe)
    save_tree(handle.create_group('warmup_recipe'),warmup_recipe)
    save_tree(handle.create_group('components'),components)
    save_tree(handle.create_group('errors'),errors)
    save_tree(handle.create_group('detector_errors'),detector_errors)
    save_tree(handle.create_group('paper_analysis'),paper_analysis)
    handle.attrs['warmup_signature']=warmup_signature
fig.savefig(OUTPUT.with_suffix('.pdf'),bbox_inches='tight')
print(OUTPUT)
